# Summarizer Evaluation

Compares the current `facebook/bart-large-cnn` summarizer against
`human-centered-summarization/financial-summarization-pegasus` on a sample of
financial news articles from the repository.

**Metrics**
- ROUGE-1 / ROUGE-2 / ROUGE-L F1 — n-gram overlap with the lead-paragraph gold proxy
- Label agreement rate — fraction of articles where the FinBERT sentiment label is
  preserved after summarization (measures information distortion)
- Wall-clock time per article

**Decision rule:** switch to financial-pegasus if it achieves ≥ 2 ROUGE-2 point
improvement **or** ≥ 3% label agreement improvement on this sample.

In [ ]:
from sentiment.log import setup_logging
setup_logging()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from sentiment.sources.news.repository import ArticleRepository
from sentiment.embeddings.summarizer import Summarizer
from sentiment.embeddings.encoder import SentimentEncoder
from sentiment.embeddings.summarizer_eval import evaluate_rouge, label_agreement_rate

# ── config ────────────────────────────────────────────────────────────────────
TICKER        = "AAPL"
SAMPLE_YEAR   = 2025
SAMPLE_MONTH  = 1
N_ARTICLES    = 50      # set lower to go faster; higher for more reliable estimates
DEVICE        = "cpu"   # change to "cuda" if available

BART_MODEL    = "facebook/bart-large-cnn"
PEGASUS_MODEL = "human-centered-summarization/financial-summarization-pegasus"

## 1. Load article sample

In [ ]:
repo = ArticleRepository()

# Load articles for one month — adjust year/month/ticker as needed
month_df = repo.read_month(SAMPLE_YEAR, SAMPLE_MONTH)
if month_df.empty:
    raise RuntimeError(f"No articles indexed for {SAMPLE_YEAR}-{SAMPLE_MONTH:02d} — run news.ipynb first")

# Filter to ticker
exploded = month_df[["id", "tickers"]].explode("tickers")
ticker_ids = set(exploded[exploded["tickers"] == TICKER]["id"])
ticker_df = month_df[month_df["id"].isin(ticker_ids)]
print(f"Articles indexed for {TICKER} in {SAMPLE_YEAR}-{SAMPLE_MONTH:02d}: {len(ticker_df)}")

articles = ticker_df[ticker_df["text"].notna() & (ticker_df["text"] != "")].head(N_ARTICLES).to_dict("records")
print(f"Loaded {len(articles)} articles with text")

## 2. Gold summary proxy

Financial newswire articles follow an inverted-pyramid structure: the most
important information appears in the lead paragraph.  We use the first
non-empty paragraph as the gold reference for ROUGE scoring.

In [ ]:
def first_paragraph(article) -> str:
    text = (article.get("text") or "").strip()
    for para in text.split("\n\n"):
        para = para.strip()
        if len(para) > 40:   # skip very short lines / headings
            return para
    return text[:300]  # fallback: first 300 chars

# Spot-check the gold extraction
for i, a in enumerate(articles[:2]):
    print(f"--- Article {i+1} gold ---")
    print(first_paragraph(a)[:200])
    print()

## 3. Load models

Both summarizers + one shared FinBERT encoder (for label agreement rate).

In [ ]:
print("Loading BART-CNN ...")
bart = Summarizer(device=DEVICE, model_name=BART_MODEL)

print("Loading financial-pegasus ...")
pegasus = Summarizer(device=DEVICE, model_name=PEGASUS_MODEL)

print("Loading FinBERT encoder ...")
encoder = SentimentEncoder(device=DEVICE)

print("All models loaded.")

## 4. ROUGE evaluation

In [ ]:
print("Evaluating BART-CNN ...")
bart_rouge = evaluate_rouge(bart, articles, first_paragraph)

print("Evaluating financial-pegasus ...")
peg_rouge = evaluate_rouge(pegasus, articles, first_paragraph)

rouge_df = pd.DataFrame({
    "BART-CNN": bart_rouge,
    "financial-pegasus": peg_rouge,
}).T[["rouge1_f", "rouge2_f", "rougeL_f", "bypass_rate", "mean_seconds_per_article", "n_articles"]]

rouge_df.columns = ["ROUGE-1 F1", "ROUGE-2 F1", "ROUGE-L F1", "Bypass rate", "Sec/article", "N"]
rouge_df = rouge_df.round(4)

print("\nROUGE scores (higher = better; gold = lead paragraph)")
display(rouge_df)

## 5. Label agreement rate

In [ ]:
print("Computing label agreement for BART-CNN ...")
bart_agree = label_agreement_rate(encoder, bart, articles)

print("Computing label agreement for financial-pegasus ...")
peg_agree = label_agreement_rate(encoder, pegasus, articles)

agree_df = pd.DataFrame({
    "BART-CNN":          {"label_agreement_rate": round(bart_agree, 4)},
    "financial-pegasus": {"label_agreement_rate": round(peg_agree, 4)},
}).T

print("\nLabel agreement rate (fraction of articles where FinBERT class is preserved)")
display(agree_df)

## 6. Side-by-side examples

In [ ]:
N_EXAMPLES = 3

for i, article in enumerate(articles[:N_EXAMPLES]):
    content = (article.get("text") or "").strip()
    title   = (article.get("title") or "").strip()
    gold    = first_paragraph(article)

    bart_sum = bart.summarize(content)
    peg_sum  = pegasus.summarize(content)

    _, _, bart_probs = encoder.encode(bart_sum)
    _, _, peg_probs  = encoder.encode(peg_sum)

    print(f"{'='*80}")
    print(f"Article {i+1}: {title[:80]}")
    print(f"{'─'*80}")
    print(f"GOLD        : {gold[:200]}")
    print(f"{'─'*80}")
    print(f"BART-CNN    : {bart_sum[:200]}")
    print(f"  FinBERT   : pos={bart_probs[0]:.3f}  neg={bart_probs[1]:.3f}  neu={bart_probs[2]:.3f}")
    print(f"{'─'*80}")
    print(f"fin-pegasus : {peg_sum[:200]}")
    print(f"  FinBERT   : pos={peg_probs[0]:.3f}  neg={peg_probs[1]:.3f}  neu={peg_probs[2]:.3f}")
    print()

## 7. Summary chart

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

rouge_metrics = ["rouge1_f", "rouge2_f", "rougeL_f"]
labels = ["ROUGE-1", "ROUGE-2", "ROUGE-L"]
x = range(len(rouge_metrics))
width = 0.35

ax = axes[0]
ax.bar([v - width/2 for v in x], [bart_rouge[m] for m in rouge_metrics], width, label="BART-CNN")
ax.bar([v + width/2 for v in x], [peg_rouge[m]  for m in rouge_metrics], width, label="financial-pegasus")
ax.set_xticks(list(x))
ax.set_xticklabels(labels)
ax.set_title("ROUGE F1 scores")
ax.set_ylabel("F1")
ax.legend()
ax.grid(True, axis="y", alpha=0.3)

ax = axes[1]
ax.bar(["BART-CNN", "financial-pegasus"], [bart_agree, peg_agree], color=["C0", "C1"])
ax.set_title("Label agreement rate")
ax.set_ylabel("Fraction")
ax.set_ylim(0, 1)
ax.grid(True, axis="y", alpha=0.3)

ax = axes[2]
ax.bar(["BART-CNN", "financial-pegasus"],
       [bart_rouge["mean_seconds_per_article"], peg_rouge["mean_seconds_per_article"]],
       color=["C0", "C1"])
ax.set_title("Wall-clock time per article")
ax.set_ylabel("Seconds")
ax.grid(True, axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Decision

Switch the default summarizer to `financial-summarization-pegasus` in
`SentimentPipeline` if **either** condition below is met:

| Condition | Threshold |
|---|---|
| ROUGE-2 F1 improvement | ≥ 0.02 (2 points) |
| Label agreement improvement | ≥ 0.03 (3%) |

In [ ]:
rouge2_gain  = peg_rouge["rouge2_f"] - bart_rouge["rouge2_f"]
agree_gain   = peg_agree - bart_agree

switch = (rouge2_gain >= 0.02) or (agree_gain >= 0.03)

print(f"ROUGE-2 gain   : {rouge2_gain:+.4f}  (threshold ≥ 0.02)")
print(f"Agreement gain : {agree_gain:+.4f}  (threshold ≥ 0.03)")
print()
if switch:
    print("RECOMMENDATION: switch to financial-summarization-pegasus")
    print("  → pass summarizer_model='human-centered-summarization/financial-summarization-pegasus'")
    print("    to SentimentPipeline()")
else:
    print("RECOMMENDATION: keep BART-CNN (no significant improvement found)")